In [10]:
import sys
!{sys.executable} -m pip install "crewai==1.14.0"

  Using cached crewai-1.14.0-py3-none-any.whl.metadata (36 kB)
Reason for being yanked: Error on dependency with anthropic
Using cached crewai-1.14.0-py3-none-any.whl (1.0 MB)
  Attempting uninstall: crewai
    Found existing installation: crewai 1.12.0
    Uninstalling crewai-1.12.0:
      Successfully uninstalled crewai-1.12.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
crypto-bot 0.1.0 requires crewai<2.0.0,>=1.15.6, but you have crewai 1.14.0 which is incompatible.
crewai-tools 1.15.6 requires crewai==1.15.6, but you have crewai 1.14.0 which is incompatible.
crewai-tools 1.15.6 requires tiktoken<0.13,>=0.8.0, but you have tiktoken 0.13.0 which is incompatible.


In [1]:
import os
from dotenv import load_dotenv

load_dotenv()

GROQ_API_KEY=os.getenv('GROQ_API_KEY')
TAVILY_API_KEY=os.getenv("TAVILY_API_KEY")

os.environ["GROQ_API_KEY"] = GROQ_API_KEY


In [2]:
import importlib.metadata
print(importlib.metadata.version("litellm"))

1.93.0


In [3]:
# CrewAI 라이브러리에서 필요한 클래스 가져오기
from crewai import Agent, Task, Crew, Process, LLM
import gradio as gr

#LLM
llm = LLM(model="groq/llama-3.3-70b-versatile", temperature=0, api_key=GROQ_API_KEY)

/Users/ysyseom/Library/Caches/pypoetry/virtualenvs/crypto-bot-udbg8-GW-py3.13/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
print(GROQ_API_KEY[:8] if GROQ_API_KEY else "None!")

gsk_xlP1


In [5]:
# Search Tool
from crewai_tools import TavilySearchTool
search_tool = TavilySearchTool(api_key = TAVILY_API_KEY)

In [6]:
# Agent 1: Researcher
researcher = Agent(
    role='Market Researcher',
    goal='Uncover emerging trends and investment opportunities in the cryptocurrency market.',
    backstory ='You are a groundbreaking researcher who identifies innovative trends and actionable insights',
    verbose = True,
    tools = [search_tool],
    allow_delegation = False,
    llm = llm,
    max_iter =3,
    max_rpm = 10
)

# Agent 2: Analyst
analyst = Agent(
    role='Investment Analyst',
    goal = 'Analyze cryptocurrency market data to extract actionable insights and investment leads.',
    backstory ='You are an expert analyst who draws meaningful conclusions from cryptocurrency market data.',
    verbose = True,
    allow_delegation = False,
    llm = llm
)

In [7]:
#Task

research_task = Task(description='Explore the internet to pinpoint emerging trends and potential investment opportunities in cryptocurrency.',
                     expected_output='A detailed summary of the research results',
                     agent=researcher)
analyst_task=Task(description='Analyze the provided cryptocurrency market data to extract key insights and compile a concise report.',
                  expected_output='A refined finalized investment report with actionable insights',
                  agent= analyst)

In [8]:
#Crew 구성
crypto_crew = Crew(agents=[researcher, analyst],
                   tasks= [research_task, analyst_task],
                   process = Process.sequential)

In [9]:
result = await crypto_crew.kickoff_async()

/Users/ysyseom/Library/Caches/pypoetry/virtualenvs/crypto-bot-udbg8-GW-py3.13/lib/python3.13/site-packages/pydantic/main.py:253: UserWarning: method callbacks cannot be serialized and will prevent checkpointing. Use a module-level named function instead.
  validated_self = self.__pydantic_validator__.validate_python(data, self_instance=self)


╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Market Researcher                                                                                       │
│                                                                                                                 │
│  Task: Explore the internet to pinpoint emerging trends and potential investment opportunities in               │
│  cryptocurrency.                                                                                                │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Tool tavily_search executed with result: {
  "query": "emerging trends and investment opportunities in cryptocurrency",
  "follow_up_questions": null,
  "answer": null,
  "images": [],
  "results": [
    {
      "url": "https://finance.yahoo...


╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Market Researcher                                                                                       │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  The cryptocurrency market is projected to reach USD 2.2 billion by 2026, growing at a CAGR of 7.1% during the  │
│  forecast period. Emerging economies and developed countries are expected to offer significant growth           │
│  opportunities for companies operating in the cryptocurrency market. Key trends shaping the rapidly evolving    │
│  crypto landscape include institutional adoption of Bitcoin ETFs, the mainstream rollout of CBDCs to enhance    │
│  financial inclusion, and the rapid rise of green crypto projects aligning blockchain innovation with global    │
│  sustainability goals. Investing in the digital asset ecosystem enables owners of spot crypto to potentially    │
│  strengthen the infrastructure their coins rely on, increase the adoption of crypto, further diversify their    │
│  crypto exposure, and capitalize on other emerging trends in the digital economy. The evolving crypto           │
│  landscape presents both opportunities and risks, and investors should closely monitor institutional            │
│  developments, regulatory shifts, and technological advancements to stay ahead of the curve. Diversifying       │
│  exposure through crypto-related equities, ETFs, and tokenized assets could offer a balanced approach to        │
│  capitalizing on this sector's growth. The increasing adoption of digital assets by traditional financial       │
│  institutions signals a maturation of the crypto market, and the recent approval of spot Bitcoin ETFs in the    │
│  U.S. has further legitimized crypto as an investable asset class, drawing in a wave of institutional capital.  │
│  Tokenized private assets such as commercial real estate and fractionalized shipping are emerging as practical  │
│  on-chain structures for raising and distributing capital. Options overlay strategies, such as the Purpose      │
│  Bitcoin Yield ETF and the Purpose Ether Yield ETF, are also gaining popularity.                                │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

[CrewAIEventsBus] Warning: Event pairing mismatch. 'agent_execution_completed' closed 'llm_call_started' (expected 
'agent_execution_started')

[CrewAIEventsBus] Warning: Event pairing mismatch. 'task_completed' closed 'agent_execution_started' (expected 
'task_started')

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Investment Analyst                                                                                      │
│                                                                                                                 │
│  Task: Analyze the provided cryptocurrency market data to extract key insights and compile a concise report.    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Investment Analyst                                                                                      │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  **Cryptocurrency Market Analysis Report**                                                                      │
│                                                                                                                 │
│  **Executive Summary**                                                                                          │
│                                                                                                                 │
│  The cryptocurrency market is poised for significant growth, with a projected value of USD 2.2 billion by       │
│  2026, growing at a CAGR of 7.1% during the forecast period. This report provides an in-depth analysis of the   │
│  current market trends, opportunities, and risks, and offers actionable insights for investors looking to       │
│  capitalize on the growth of the digital asset ecosystem.                                                       │
│                                                                                                                 │
│  **Market Trends and Opportunities**                                                                            │
│                                                                                                                 │
│  1. **Institutional Adoption of Bitcoin ETFs**: The recent approval of spot Bitcoin ETFs in the U.S. has        │
│  legitimized crypto as an investable asset class, drawing in a wave of institutional capital. This trend is     │
│  expected to continue, with more traditional financial institutions adopting Bitcoin ETFs and other             │
│  crypto-related products.                                                                                       │
│  2. **Mainstream Rollout of CBDCs**: The rollout of Central Bank Digital Currencies (CBDCs) is expected to      │
│  enhance financial inclusion and provide significant growth opportunities for companies operating in the        │
│  cryptocurrency market.                                                                                         │
│  3. **Rise of Green Crypto Projects**: The rapid rise of green crypto projects aligning blockchain innovation   │
│  with global sustainability goals is expected to drive growth and adoption in the cryptocurrency market.        │
│  4. **Tokenized Private Assets**: Tokenized private assets such as commercial real estate and fractionalized    │
│  shipping are emerging as practical on-chain structures for raising and distributing capital.                   │
│  5. **Options Overlay Strategies**: Options overlay strategies, such as the Purpose Bitcoin Yield ETF and the   │
│  Purpose Ether Yield ETF, are gaining popularity and offer investors a new way to capitalize on the growth of   │
│  the digital asset ecosystem.                                                                                   │
│                                                                                                                 │
│  **Investment Opportunities**                                                                                   │
│                                                                                                                 │
│  1. **Diversifying Exposure**: Diversifying exposure through crypto-related equities, ETFs, and tokenized       │
│  assets could offer a balanced approach to capitalizing

[CrewAIEventsBus] Warning: Event pairing mismatch. 'crew_kickoff_completed' closed 'task_started' (expected 
'crew_kickoff_started')